## Chaotic scenario 3: Extreme volatility

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from amm_basics import *

# 1. DATA ACQUISITION & SIMULATION SETUP

n_points = 1440
start_price = 1600
initial_variance = 5
variance_growth_rate = 0.01

np.random.seed(42) 

variances = initial_variance * np.exp(variance_growth_rate * np.arange(n_points))

steps = np.random.normal(0, np.sqrt(variances))

prices = start_price + np.cumsum(steps)

timestamps = pd.date_range(start=pd.Timestamp.now() - pd.Timedelta(days=1),
                           periods=n_points, freq='T')

df = pd.DataFrame({"timestamp": timestamps, "price_dai": prices})

eth, dai = AtomicToken("ETH"), AtomicToken("DAI")
arb_wallet = Wallet("Arbitrageur")
trader_wallet = Wallet("NoiseTraders")

initial_eth, initial_dai = 1000.0, 1000000.0
arb_wallet.deposit(eth, initial_eth)
arb_wallet.deposit(dai, initial_dai)
trader_wallet.deposit(eth, 1000.0)
trader_wallet.deposit(dai, 2500000.0)


frame_duration = 10     
transition_duration = 10 
fee = 0.01  
step_skip = 5
p_init = df["price_dai"].iloc[0]
x_start = 10.0

amm_pool = AMM(eth, dai, UniswapV2(), reserve0=x_start, reserve1=x_start * p_init)
sim_state = State([arb_wallet, trader_wallet], [amm_pool])

# 2. SIMULATION EXECUTION WITH IL & FEES

history = []

initial_eth_pool, initial_dai_pool = amm_pool.reserves[eth], amm_pool.reserves[dai]
total_fees = 0.0

for i in range(len(df)):
    m_price = df["price_dai"].iloc[i]
    res = amm_pool.reserves

    # Noise Trades (Market Swings)
    if np.random.random() < 0.5:
        direction = np.random.choice(["buy", "sell"])
        if direction == "buy":
            amount_dai = res[dai] * np.random.uniform(0.0005, 0.005)
            sim_state.swap(Transaction("swap", trader_wallet, dai, eth, amount_dai))
            total_fees += amount_dai * fee  # sumar fees en DAI
        else:
            amount_eth = res[eth] * np.random.uniform(0.0005, 0.005)
            sim_state.swap(Transaction("swap", trader_wallet, eth, dai, amount_eth))
            total_fees += amount_eth * fee * m_price  # convertir fees a DAI

    # Arbitrage Logic
    res = amm_pool.reserves
    p_amm = res[dai] / res[eth]
    p_bid, p_ask = p_amm * (1 - fee), p_amm / (1 - fee)
    
    if m_price > p_ask:
        target_x = np.sqrt((res[eth]*res[dai]) / (m_price * (1-fee)))
        dy_in = (res[eth]*res[dai] / target_x) - res[dai]
        if dy_in > 0.001: 
            sim_state.swap(Transaction("swap", arb_wallet, dai, eth, dy_in))
    elif m_price < p_bid:
        target_x = np.sqrt((res[eth]*res[dai]) / (m_price / (1-fee)))
        dx_in = target_x - res[eth]
        if dx_in > 0.001: 
            sim_state.swap(Transaction("swap", arb_wallet, eth, dai, dx_in))

    res = amm_pool.reserves
    current_k = res[eth] * res[dai]
    m_x = np.sqrt(current_k / m_price)

    theta = (res[dai] / res[eth]) / p_init  # relación del pool vs precio inicial
    il_val = (2 * np.sqrt(theta) / (1 + theta) - 1)  # fórmula clásica de IL

    # Net Return
    v_pool = res[eth] * m_price + res[dai]
    v_hodl = initial_eth_pool * m_price + initial_dai_pool
    net_val = (v_pool / v_hodl) - 1

    history.append({
        'x': res[eth], 'y': res[dai], 'k': current_k,
        'm_x': m_x, 'm_y': current_k / m_x,
        'bid_x': res[eth] * np.sqrt(1/(1-fee)), 
        'ask_x': res[eth] * np.sqrt(1-fee),
        'IL': il_val,             # IL con θ
        'fees': total_fees,       # acumulado directamente
        'net_return': net_val     # ganancia neta desde el inicio
    })

h_df = pd.DataFrame(history)


# 3. DYNAMIC VISUALIZATION WITH IL, FEES & NET RETURN

y_min, y_max = df["price_dai"].min() * 0.97, df["price_dai"].max() * 1.03
x_time_min, x_time_max = df["timestamp"].min(), df["timestamp"].max()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("<b>Synthetic External Market Price</b>", "<b>CPMM Curve: k = x*y (Uniswap V2)</b>"),
    horizontal_spacing=0.15
)

x_range = np.linspace(h_df['x'].min()*0.8, h_df['x'].max()*1.2, 200)

# ----------------------
# INITIAL TRACES
# ----------------------
# Market Price
fig.add_trace(go.Scatter(
    x=[df["timestamp"][0]], y=[df["price_dai"][0]],
    mode='lines', line=dict(color='blue'),
    name="Market Price (DAI)",
    showlegend=False  # Visible en la gráfica pero no en la leyenda
), row=1, col=1)

# Invariant Curve
fig.add_trace(go.Scatter(
    x=x_range, y=h_df['k'][0]/x_range,
    line=dict(color='rgba(0,0,0,0.1)', width=1),
    name="Invariant Curve (k)",
    showlegend=False  # Visible en la gráfica pero no en la leyenda
), row=1, col=2)

# Ext. Market State
fig.add_trace(go.Scatter(
    x=[h_df['m_x'][0]], y=[h_df['m_y'][0]], mode='markers',
    marker=dict(color='blue', size=8),
    name="Ext. Market State",
    showlegend=False
), row=1, col=2)

# Fee Band
fig.add_trace(go.Scatter(
    x=[h_df['bid_x'][0]], y=[h_df['k'][0]/h_df['bid_x'][0]], mode='markers',
    marker=dict(color='orange', symbol='line-ns-open'),
    name="Fee Band",
    showlegend=False
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=[h_df['ask_x'][0]], y=[h_df['k'][0]/h_df['ask_x'][0]], mode='markers',
    marker=dict(color='orange', symbol='line-ns-open'),
    showlegend=False
), row=1, col=2)

# Pool State
fig.add_trace(go.Scatter(
    x=[h_df['x'][0]], y=[h_df['y'][0]], mode='markers',
    marker=dict(color='red', size=12, line=dict(color='white', width=1)),
    name="Pool State",
    legendgroup="pool"
), row=1, col=2)

# IL en leyenda
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    showlegend=True,
    name=f"IL: {h_df['IL'][0]:.4%}",
    legendgroup="pool"
))

# Fees en leyenda
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    showlegend=True,
    name=f"Fees: {h_df['fees'][0]:.2f} DAI",
    legendgroup="pool"
))

# Net Return en leyenda
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    showlegend=True,
    name=f"Net Return: {h_df['net_return'][0]:.4%}",
    legendgroup="pool"
))


fig.update_xaxes(title_text="Time", row=1, col=1)
fig.update_yaxes(title_text="Price (DAI/ETH)", row=1, col=1)
fig.update_xaxes(title_text="ETH Reserves (x)", row=1, col=2)
fig.update_yaxes(title_text="DAI Reserves (y)", row=1, col=2)


frames = []
for i in range(0, len(df), step_skip):

    x_min, x_max = h_df['x'][:i+1].min()*0.95, h_df['x'][:i+1].max()*1.05
    y_min_dyn, y_max_dyn = h_df['y'][:i+1].min()*0.95, h_df['y'][:i+1].max()*1.05
    
    frames.append(go.Frame(
        data=[
            # Market Price
            go.Scatter(x=df["timestamp"][:i+1], y=df["price_dai"][:i+1]),
            # Invariant Curve
            go.Scatter(x=x_range, y=h_df['k'][i]/x_range),
            # Ext. Market State
            go.Scatter(x=[h_df['m_x'][i]], y=[h_df['m_y'][i]]),
            # Fee Band
            go.Scatter(x=[h_df['bid_x'][i]], y=[h_df['k'][i]/h_df['bid_x'][i]]),
            go.Scatter(x=[h_df['ask_x'][i]], y=[h_df['k'][i]/h_df['ask_x'][i]]),
            # Pool State
            go.Scatter(x=[h_df['x'][i]], y=[h_df['y'][i]]),
            # IL, Fees y Net Return en frames
            go.Scatter(x=[None], y=[None], name=f"IL: {h_df['IL'][i]:.4%}"),
            go.Scatter(x=[None], y=[None], name=f"Fees: {h_df['fees'][i]:.2f} DAI"),
            go.Scatter(x=[None], y=[None], name=f"Net Return: {h_df['net_return'][i]:.4%}")
        ],
        name=f"f{i}",
        layout=go.Layout(
            xaxis2=dict(range=[x_min, x_max]),
            yaxis2=dict(range=[y_min_dyn, y_max_dyn])
        )
    ))


fig.frames = frames


fig.update_layout(
    height=600, template="plotly_white",
    xaxis1=dict(range=[x_time_min, x_time_max], type="date", fixedrange=True),
    yaxis1=dict(range=[y_min, y_max], fixedrange=True),
    xaxis2=dict(range=[h_df['x'].min()*0.95, h_df['x'].max()*1.05]),
    yaxis2=dict(range=[h_df['y'].min()*0.95, h_df['y'].max()*1.05]),
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        x=0.0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        direction="right",
        buttons=[dict(
            label="▶ Start Simulation",
            method="animate",
            args=[None, {
                "frame": {"duration": 0.5, "redraw": True},
                "fromcurrent": True,
                "transition": {"duration": transition_duration, "easing": "quadratic-in-out"},
                "mode": "immediate"
            }]
        )]
    )]
)

fig.show()


![Volatility Simulation](volatility_simulation.gif)

In the animation, we observe that when the token exhibits high volatility, the environment becomes highly chaotic for Liquidity Providers (LPs). As reflected in the performance metrics, the LP’s net return is approximately -56%. This implies that the LP has lost more than half of their value compared to a simple buy-and-hold strategy.

### Theorical background

The time series of the simulation cames from the following stochastic process:

$$
X_t = S_0 + \sum_{k=1}^{t} Z_k,
$$

where

$$
Z_k \sim \mathcal{N}(0, \sigma_k^2),
\qquad \text{with} \qquad
\sigma_k^2 = \sigma_0^2 e^{\alpha k}.
$$

The stochastic process defined above has a variance that increases exponentially over time. As the variance grows, the dispersion of $X_t$ around its mean increases accordingly. In other words, higher variance implies that realizations of the process are more widely spread around the average value.

Since the price ratio

$$
\theta = \frac{P_{\text{final}}}{P_{\text{initial}}}
$$

depends on the terminal price, greater dispersion in $X_t$ translates into greater dispersion in $\theta$.

The impermanent loss function

$$
I(\theta) = \frac{2\sqrt{\theta}}{1 + \theta} - 1
$$

is concave and attains its maximum value of $0$ when $\theta = 1$, that is, when the final price equals the initial price. As $\theta$ moves away from 1 in either direction, $I(\theta)$ becomes increasingly negative.

Therefore, when the variance of the process is high, price realizations tend to be more dispersed and further away from the initial level. Because impermanent loss decreases as prices diverge from equality, higher variance leads to larger values of impermanent loss in absolute terms.

Finally, note that the additive specification of the process may generate negative values for $X_t$. Although this is not realistic for asset prices, this simplification is accepted here in order to focus exclusively on the effect of increasing variance on impermanent loss.

### Conclusion

The increased variance process develops an important principle: volatility has a non-neutral impact on liquidity providers. Even when there is no directional price drop, an expanding price distribution will produce progressively larger impermanent losses. This occurs because of the concavity of $I(\theta)$, which penalizes dispersion regardless of its direction. Therefore, the symmetric expansion of uncertainty in an asset market will create greater expected losses.

As a result of the above findings, there exists a practical consequence that is frequently overlooked: LPs do not have to be wrong regarding the direction of an asset market in order to experience a loss. As long as they are exposed to a highly volatile asset (which is often the case with cryptocurrencies) they will incur losses. The ability of cryptocurrency markets to produce large increases in variance in a very short period of time creates a structural disadvantage for LPs, who must continuously offset these losses through fee revenue just to break even.